In [3]:
# Phase 0 — Kiểm toán dữ liệu Steam

# Mục tiêu:
# - Xác nhận file nguồn, số dòng và số cột.
# - Kiểm tra định danh game.
# - Khảo sát kiểu dữ liệu và giá trị thiếu.
# - Ghi nhận vấn đề trước khi quyết định làm sạch.

# File trong data/raw được giữ nguyên.

In [4]:
from pathlib import Path
import pandas as pd

# Xác định thư mục gốc khi chạy từ repo hoặc notebooks.
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "steam_games.csv"

print("File đang đọc:", DATA_PATH)
print("File tồn tại:", DATA_PATH.is_file())

File đang đọc: d:\Steam Market Intelligence What Makes a Game Succeed on Steam\Steam-Market-Intelligence-What-Makes-a-Game-Succeed-on-Steam\data\raw\steam_games.csv
File tồn tại: True


In [5]:
df = pd.read_csv(DATA_PATH)

print(f"Số dòng: {df.shape[0]:,}")
print(f"Số cột: {df.shape[1]}")

display(df.head())

Số dòng: 71,429
Số cột: 21


,steam_appid,name,developers,publishers,categories,genres,required_age,n_achievements,platforms,is_released,...,additional_content,total_reviews,total_positive,total_negative,review_score,review_score_desc,positive_percentual,metacritic,is_free,price_initial (USD)
0,2719580,勇者の伝説の勇者,['ぽけそう'],['ぽけそう'],"['Single-player', 'Family Sharing']","['Casual', 'Indie']",0,0,['windows'],True,...,[],0,0,0,0.0,No user reviews,0.0,0,False,0.99
1,2719590,Light No Fire,['Hello Games'],['Hello Games'],"['Single-player', 'Multi-player', 'Co-op', 'On...","['Action', 'Adventure']",0,0,['windows'],False,...,[],0,0,0,0.0,No user reviews,0.0,0,False,0.00
2,2719600,Lorhaven: Cursed War,['GoldenGod Games'],['GoldenGod Games'],"['Single-player', 'Multi-player', 'PvP', 'Shar...","['RPG', 'Strategy']",0,32,"['windows', 'mac']",True,...,[],9,8,1,0.0,9 user reviews,88.9,0,False,9.99
3,2719610,PUIQ: Demons,['Giammnn'],['Giammnn'],"['Single-player', 'Steam Achievements', 'Famil...","['Action', 'Casual', 'Indie', 'RPG']",0,28,['windows'],True,...,[],0,0,0,0.0,No user reviews,0.0,0,False,2.99
4,2719650,Project XSTING,['Saucy Melon'],['Saucy Melon'],"['Single-player', 'Steam Achievements', 'Steam...","['Action', 'Casual', 'Indie', 'Early Access']",0,42,['windows'],True,...,[],9,9,0,0.0,9 user reviews,100.0,0,False,7.99


In [6]:
missing_ids = df["steam_appid"].isna().sum()
duplicate_ids = df["steam_appid"].duplicated().sum()

print(f"ID bị thiếu: {missing_ids}")
print(f"ID bị trùng: {duplicate_ids}")

ID bị thiếu: 0
ID bị trùng: 0


In [7]:
audit_summary = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).to_numpy(),
    "null_count": df.isna().sum().to_numpy(),
    "null_pct": (
        df.isna().mean().mul(100).round(2).to_numpy()
    )
})

display(audit_summary)

,column,dtype,null_count,null_pct
0,steam_appid,int64,0,0.0
1,name,object,0,0.0
2,developers,object,0,0.0
3,publishers,object,0,0.0
4,categories,object,0,0.0
5,genres,object,0,0.0
6,required_age,int64,0,0.0
7,n_achievements,int64,0,0.0
8,platforms,object,0,0.0
9,is_released,bool,0,0.0


In [8]:
## Kiểm tra các trường dạng danh sách

# Mục tiêu: xem cách lưu dữ liệu và đếm danh sách rỗng.
# Chưa thay thế giá trị hoặc xóa dòng.

In [9]:
array_columns = [
    "developers",
    "publishers",
    "categories",
    "genres",
    "platforms",
    "additional_content"
]

# Xem mẫu và kiểu của một giá trị.
display(df[array_columns].head(3))

print("Kiểu giá trị genres:", type(df["genres"].iloc[0]))

,developers,publishers,categories,genres,platforms,additional_content
0,['ぽけそう'],['ぽけそう'],"['Single-player', 'Family Sharing']","['Casual', 'Indie']",['windows'],[]
1,['Hello Games'],['Hello Games'],"['Single-player', 'Multi-player', 'Co-op', 'On...","['Action', 'Adventure']",['windows'],[]
2,['GoldenGod Games'],['GoldenGod Games'],"['Single-player', 'Multi-player', 'PvP', 'Shar...","['RPG', 'Strategy']","['windows', 'mac']",[]


Kiểu giá trị genres: <class 'str'>


In [10]:
array_audit = []

for col in array_columns:
    values = df[col].astype("string").str.strip()

    array_audit.append({
        "column": col,
        "null_count": int(values.isna().sum()),
        "blank_count": int(values.eq("").sum()),
        "empty_list_count": int(values.eq("[]").sum()),
        "empty_list_pct": round(values.eq("[]").mean() * 100, 2)
    })

array_audit = pd.DataFrame(array_audit)

display(array_audit)

,column,null_count,blank_count,empty_list_count,empty_list_pct
0,developers,0,0,200,0.28
1,publishers,0,0,493,0.69
2,categories,0,0,558,0.78
3,genres,0,0,223,0.31
4,platforms,0,0,0,0.00
5,additional_content,0,0,64266,89.97


In [11]:
## Kiểm tra tính nhất quán của review

# Kiểm tra giá trị âm, tổng review và số game không có review.
# Chưa loại game khỏi dữ liệu.

In [12]:
review_columns = [
    "total_reviews",
    "total_positive",
    "total_negative"
]   
print(df[review_columns].dtypes)

total_reviews     int64
total_positive    int64
total_negative    int64
dtype: object


In [13]:
## Review theo trạng thái phát hành

# Phân biệt game đã phát hành và chưa phát hành nhằm xác định
# phạm vi phù hợp cho việc đánh giá mức độ tương tác.
# Tỷ lệ không có review được tính riêng trong từng nhóm.

In [14]:
release_review_audit = (
    df.groupby("is_released")
    .agg(
        total_games=("steam_appid", "size"),
        zero_review_games=("total_reviews", lambda reviews: (reviews == 0).sum()),
        qualified_games=("total_reviews", lambda reviews: (reviews >= 50).sum()),
        median_reviews=("total_reviews", "median")
    )
)

release_review_audit["zero_review_pct"] = ( release_review_audit["zero_review_games"] / release_review_audit["total_games"] * 100).round(2)
release_review_audit["qualified_pct"] = (release_review_audit["qualified_games"] / release_review_audit["total_games"] * 100).round(2)

display(release_review_audit)

,total_games,zero_review_games,qualified_games,median_reviews,zero_review_pct,qualified_pct
is_released,,,,,,
False,13388,13388,0,0.0,100.00,0.00
True,58041,13424,11568,5.0,23.13,19.93


In [15]:
# Xem mẫu ngày của game đã phát hành.
print("Mẫu release_date của game đã phát hành:")

display(
    df.loc[df["is_released"], "release_date"]
    .drop_duplicates()
    .head(10)
)

# Kiểm tra toàn bộ nhãn ngày ở nhóm chưa phát hành.
print("Giá trị release_date của game chưa phát hành:")

display(
    df.loc[~df["is_released"], "release_date"]
    .value_counts(dropna=False)
)

Mẫu release_date của game đã phát hành:


0     2024-01-04
2     2024-01-26
3     2024-02-17
4     2024-01-05
7     2024-01-15
8     2024-03-26
12    2024-02-29
13    2024-05-23
14    2024-06-08
16    2024-04-06
Name: release_date, dtype: object

Giá trị release_date của game chưa phát hành:


release_date
Not Released    13388
Name: count, dtype: int64

In [16]:
release_dates = df["release_date"].astype("string").str.strip()

# Nhận diện hai cấu trúc đang quan sát được.
date_only = release_dates.str.fullmatch(
    r"\d{4}-\d{2}-\d{2}", na=False
)

date_with_time = release_dates.str.fullmatch(
    r"\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}", na=False
)

# Tạo Series datetime riêng, giữ nguyên cột nguồn.
parsed_dates = pd.Series(pd.NaT, index=df.index, dtype="datetime64[ns]")

parsed_dates.loc[date_only] = pd.to_datetime(
    release_dates.loc[date_only],
    format="%Y-%m-%d",
    errors="coerce"
)

parsed_dates.loc[date_with_time] = pd.to_datetime(
    release_dates.loc[date_with_time],
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)

released_invalid_date = df["is_released"] & parsed_dates.isna()
unreleased_unexpected_label = (~df["is_released"] & release_dates.ne("Not Released"))

print("Số giá trị dạng ngày:", date_only.sum())
print("Số giá trị dạng ngày và giờ:", date_with_time.sum())
print("Game đã phát hành chưa chuyển đổi được ngày:", released_invalid_date.sum())
print("Game chưa phát hành có nhãn khác Not Released:", unreleased_unexpected_label.sum())

valid_dates = parsed_dates.loc[df["is_released"]].dropna()

print("Ngày phát hành sớm nhất:", valid_dates.min())
print("Ngày phát hành muộn nhất:", valid_dates.max())

date_issues = released_invalid_date | unreleased_unexpected_label

if date_issues.any():
    display(
        df.loc[
            date_issues,
            ["steam_appid", "name", "release_date"]
        ].head(10)
    )

Số giá trị dạng ngày: 28
Số giá trị dạng ngày và giờ: 58013
Game đã phát hành chưa chuyển đổi được ngày: 0
Game chưa phát hành có nhãn khác Not Released: 0
Ngày phát hành sớm nhất: 1997-06-30 00:00:00
Ngày phát hành muộn nhất: 2025-01-23 00:00:00


In [17]:
price = df["price_initial (USD)"]

print("Số game có giá âm:", (price < 0).sum())
print("Giá thấp nhất:", price.min())
print("Giá cao nhất:", price.max())
print("Giá trung bình:", round(price.mean(), 2))
print("Giá trung vị:", price.median())

price_status = pd.Series(
    "Không xác định", index=df.index, name="price_status"
)

price_status.loc[price == 0] = "Giá bằng 0"
price_status.loc[price > 0] = "Giá dương"
price_status.loc[price < 0] = "Giá âm"

display(
    pd.crosstab(
        index=[df["is_released"], df["is_free"]],
        columns=price_status
    )
)

Số game có giá âm: 0
Giá thấp nhất: 0.0
Giá cao nhất: 999.98
Giá trung bình: 6.53
Giá trung vị: 2.99


price_status         Giá bằng 0  Giá dương
is_released is_free                       
False       False         12153         22
            True           1213          0
True        False           740      48970
            True           8328          3

In [18]:
price = df["price_initial (USD)"]
released = df["is_released"]

# Hai trường hợp không nhất quán trong nhóm đã phát hành.
paid_zero = released & ~df["is_free"] & price.eq(0)
free_positive = released & df["is_free"] & price.gt(0)

columns_to_check = [
    "steam_appid",
    "name",
    "is_released",
    "is_free",
    "price_initial (USD)"
]

print("Game đã ra mắt ghi nhận miễn phí nhưng có giá > 0:")
display(df.loc[free_positive, columns_to_check].head(10))

print("Mẫu game đã ra mắt ghi nhận trả phí nhưng giá = 0:")
display(df.loc[paid_zero, columns_to_check].head(10))

print("10 game đã phát hành có giá cao nhất:")
display(
    df.loc[released, columns_to_check]
    .nlargest(10, "price_initial (USD)")
)

Game đã ra mắt ghi nhận miễn phí nhưng có giá > 0:


,steam_appid,name,is_released,is_free,price_initial (USD)
41487,481870,Mini Thief,True,True,4.99
54648,397720,Romance of the Three Kingdoms Maker,True,True,9.99
65813,608990,The Archotek Project,True,True,0.99


Mẫu game đã ra mắt ghi nhận trả phí nhưng giá = 0:


,steam_appid,name,is_released,is_free,price_initial (USD)
1308,2677280,梦境世界,True,False,0.0
2148,2651680,Kuudere Funk,True,False,0.0
2191,2650750,Echoes of Fear,True,False,0.0
2917,2628680,Grill on Wheels,True,False,0.0
2932,2628100,Swords of Blood,True,False,0.0
2951,2627730,i Pick Skills,True,False,0.0
4341,2584160,AlphaBit,True,False,0.0
4410,2581540,Magic And Dragons : 4X Battle,True,False,0.0
5265,2554820,Celebrities Hacked,True,False,0.0
5590,2544880,Dreamland,True,False,0.0


10 game đã phát hành có giá cao nhất:


,steam_appid,name,is_released,is_free,price_initial (USD)
6979,2504210,The Leverage Game Business Edition,True,False,999.98
69053,1105450,Aartform Curvy 3D 4.0,True,False,299.99
25764,253670,Aartform Curvy 3D 3.0,True,False,299.90
61603,502570,Houdini Indie,True,False,269.99
53609,438450,3DF Zephyr Lite Steam Edition,True,False,249.00
495,2703200,Hidden Western Top-Down 3D,True,False,199.99
496,2703210,3D PUZZLE - Wild West,True,False,199.99
497,2703220,3D PUZZLE - World War II,True,False,199.99
498,2703240,Scream or Die - Virtual Circus,True,False,199.99
499,2703250,3D PUZZLE - Breaking Bed,True,False,199.99


In [19]:
from ast import literal_eval 
# literal_eval đọc một chuỗi và chuyển nó thành đối tượng Python thật, một cách an toàn

# Chuyển chuỗi biểu diễn danh sách thành danh sách Python.
def parse_genres(value):
    try:
        genres = literal_eval(value)
    except (ValueError, SyntaxError, TypeError):
        return None
    # ValueError, SyntaxError: chuỗi không đúng cú pháp Python.
    # TypeError: giá trị không phải chuỗi (ví dụ NaN).
    if isinstance(genres, list) and all(
        isinstance(item, str) for item in genres
    ):
        return genres

    return None


released_products = df.loc[df["is_released"],["steam_appid", "name", "genres", "price_initial (USD)"]].copy()
released_products["genre_list"] = released_products["genres"].map(parse_genres)

parse_failed = released_products["genre_list"].isna()
print("Số bản ghi không đọc được danh sách genre:", parse_failed.sum())

if parse_failed.any():
    display(released_products.loc[parse_failed, ["steam_appid", "genres"]].head(5))

# Xem nhãn thể loại của 10 sản phẩm có giá cao nhất.
display(
    released_products.sort_values(by="price_initial (USD)", ascending=False)
    [["steam_appid", "name", "price_initial (USD)", "genre_list"]]
    .head(10)
)

# Đếm số sản phẩm có từng nhãn thể loại.
genre_counts = (
    released_products[["steam_appid", "genre_list"]]
    .explode("genre_list")
    .dropna(subset=["genre_list"])
    .drop_duplicates(subset=["steam_appid", "genre_list"])
    ["genre_list"].value_counts()
)

display(genre_counts.rename("product_count").to_frame())
# value_counts() trả về Series. .rename("product_count") đặt tên cột đếm, .to_frame() đổi thành DataFrame để display hiện thành bảng đẹp.

Số bản ghi không đọc được danh sách genre: 0


,steam_appid,name,price_initial (USD),genre_list
6979,2504210,The Leverage Game Business Edition,999.98,"[Indie, Simulation]"
69053,1105450,Aartform Curvy 3D 4.0,299.99,"[Animation & Modeling, Design & Illustration]"
25764,253670,Aartform Curvy 3D 3.0,299.90,[Animation & Modeling]
61603,502570,Houdini Indie,269.99,"[Animation & Modeling, Design & Illustration, ..."
53609,438450,3DF Zephyr Lite Steam Edition,249.00,"[Animation & Modeling, Design & Illustration, ..."
44798,1607850,Planets Battle,199.99,"[Action, Adventure, Casual, Indie, Simulation,..."
29686,2818040,奇境绿洲,199.99,"[Adventure, Casual, Indie, Racing]"
44799,1607860,Water Drift,199.99,"[Action, Adventure, Casual, Indie, Racing, RPG..."
57938,2973540,3D PUZZLE - Hospital 2,199.99,"[Action, Adventure, Casual, Indie, Racing, RPG..."
57939,2973550,3D PUZZLE - Hospital 3,199.99,"[Action, Adventure, Casual, Indie, Racing, RPG..."


,product_count
genre_list,
Indie,41028
Casual,24264
Action,24129
Adventure,23079
Simulation,11801
Strategy,11279
RPG,10725
Early Access,6118
Free To Play,5155


In [20]:
# Nhãn ứng viên để nhận diện phần mềm hoặc nội dung video.
# Đây là quy tắc phân loại của project, cần ghi lại trong tài liệu.
non_game_labels = {
    "Utilities",
    "Design & Illustration",
    "Animation & Modeling",
    "Video Production",
    "Game Development",
    "Audio Production",
    "Software Training",
    "Photo Editing",
    "Web Publishing",
    "Accounting",
    "Movie",
    "Documentary",
    "360 Video"
}

# Các nhãn game dùng để kiểm tra trường hợp có cả hai nhóm.
game_labels = {
    "Action",
    "Adventure",
    "Casual",
    "Indie",
    "Simulation",
    "Strategy",
    "RPG",
    "Sports",
    "Racing",
    "Massively Multiplayer"
}

released_products["has_non_game_label"] = (
    released_products["genre_list"]
    .map(lambda genres: bool(set(genres) & non_game_labels))
)

released_products["has_game_label"] = (
    released_products["genre_list"]
    .map(lambda genres: bool(set(genres) & game_labels))
)

print("Số sản phẩm theo hai nhóm nhãn:")

display(
    pd.crosstab(
        released_products["has_non_game_label"],
        released_products["has_game_label"],
        margins=True #Tự động cộng dồn tổng số lượng theo hàng và theo cột
    )
)

print("Mẫu sản phẩm có cả nhãn game và nhãn phần mềm/video:")

mixed_labels = (
    released_products["has_non_game_label"] & released_products["has_game_label"]
)

display(
    released_products.loc[mixed_labels,["steam_appid", "name", "genre_list"]].head(15)
)

Số sản phẩm theo hai nhóm nhãn:


has_game_label,False,True,All
has_non_game_label,,,
False,388,56788,57176
True,707,158,865
All,1095,56946,58041


Mẫu sản phẩm có cả nhãn game và nhãn phần mềm/video:


,steam_appid,name,genre_list
675,2696730,virtual dream,"[Casual, Indie, Simulation, Design & Illustrat..."
779,2693120,XBPlay,"[Adventure, Casual, Indie, Utilities]"
907,2689770,Robot Dragon Eagle,"[Action, Adventure, Indie, Game Development]"
1646,2667350,Anemis: Desktop Animator,"[Casual, Simulation, Animation & Modeling]"
1701,2665370,VIRTUAL LIVE ARENA,"[Casual, Free To Play, Audio Production, Softw..."
1796,2662620,◒ Art Boxer,"[Casual, Design & Illustration, Photo Editing,..."
2224,2650600,大话商周,"[RPG, Game Development]"
2521,2641350,YoloX - Crosshair Overlay,"[Action, Animation & Modeling, Design & Illust..."
3653,2603950,Timeline 拯救存档 -Manage your game save,"[Indie, Education, Photo Editing, Software Tra..."
3761,2601130,Etelä-Pohjanmaa Experience,"[Casual, Utilities]"


In [21]:
released_products["scope_group"] = "Chưa xác định"

game_only = (
    released_products["has_game_label"]
    & ~released_products["has_non_game_label"]
)

non_game_only = (
    ~released_products["has_game_label"]
    & released_products["has_non_game_label"]
)

mixed = (
    released_products["has_game_label"]
    & released_products["has_non_game_label"]
)

released_products.loc[game_only, "scope_group"] = "Mẫu game chính"
released_products.loc[non_game_only, "scope_group"] = "Phần mềm hoặc video"
released_products.loc[mixed, "scope_group"] = "Nhãn giao thoa"

scope_summary = (
    released_products["scope_group"]
    .value_counts()
    .rename_axis("scope_group")
    .to_frame("product_count")
)

scope_summary["pct"] = (
    scope_summary["product_count"] / len(released_products) * 100
).round(2)

display(scope_summary)

,product_count,pct
scope_group,,
Mẫu game chính,56788,97.84
Phần mềm hoặc video,707,1.22
Chưa xác định,388,0.67
Nhãn giao thoa,158,0.27


In [22]:
## Kiểm tra cấu trúc các trường dạng danh sách
# Kiểm tra khả năng chuyển đổi chuỗi thành danh sách, kiểu
# phần tử, phần tử rỗng và phần tử trùng trong từng bản ghi.
# Phạm vi kiểm tra gồm toàn bộ dữ liệu
array_columns = [
    "developers",
    "publishers",
    "categories",
    "genres",
    "platforms",
    "additional_content"
]

def parse_array(value):
    try:
        result = literal_eval(value)
    except (ValueError, SyntaxError, TypeError):
        return None

    if isinstance(result, list):
        return result

    return None


array_structure_audit = []
array_issue_examples = []

for col in array_columns:
    parsed = df[col].map(parse_array)

    parse_failed = parsed.isna()

    wrong_element_type = parsed.map(
        lambda items: isinstance(items, list)
        and any(not isinstance(item, str) for item in items)
    )

    blank_element = parsed.map(
        lambda items: isinstance(items, list)
        and any(
            isinstance(item, str) and item.strip() == ""
            for item in items
        )
    )

    # Chỉ kiểm tra trùng khi mọi phần tử đều là chuỗi.
    duplicate_element = parsed.map(
        lambda items: isinstance(items, list)
        and all(isinstance(item, str) for item in items)
        and len(items) != len(set(items))
    )

    array_structure_audit.append({
        "column": col,
        "parse_failed": int(parse_failed.sum()),
        "wrong_element_type": int(wrong_element_type.sum()),
        "blank_element": int(blank_element.sum()),
        "duplicate_element": int(duplicate_element.sum())
    })

    has_issue = (
        parse_failed
        | wrong_element_type
        | blank_element
        | duplicate_element
    )

    if has_issue.any():
        examples = df.loc[
            has_issue, ["steam_appid", col]
        ].head(3)

        for _, row in examples.iterrows():
            array_issue_examples.append({
                "column": col,
                "steam_appid": row["steam_appid"],
                "value_excerpt": str(row[col])[:200]
            })

display(pd.DataFrame(array_structure_audit))

if array_issue_examples:
    display(pd.DataFrame(array_issue_examples))

,column,parse_failed,wrong_element_type,blank_element,duplicate_element
0,developers,0,0,1,53
1,publishers,0,0,6,13
2,categories,0,0,0,67
3,genres,0,0,0,0
4,platforms,0,0,0,0
5,additional_content,0,0,0,8


,column,steam_appid,value_excerpt
0,developers,2696930,"['Rising Peaks Studio', 'Rising Peaks Studio']"
1,developers,2687390,"['Greenlynx', 'DreamMasters Studio', 'Greenlynx']"
2,developers,2592550,"['RefractionGames', '闪晃', '闪晃']"
3,publishers,2592550,"['闪晃', '闪晃']"
4,publishers,2560620,"['Eid Games', 'Eid Games']"
5,publishers,2521400,"['Heretic Ceramics', 'Heretic Ceramics']"
6,categories,2659710,"['Single-player', 'Multi-player', 'PvP', 'Onli..."
7,categories,2610250,"['Single-player', 'Steam Workshop', 'Steam Wor..."
8,categories,2586630,"['Single-player', 'Steam Achievements', 'Full ..."
9,additional_content,923140,"['Elven Magic SE', 'Elven Magic 2', 'Elven Mag..."
